## Lifelines Survival Modes With Canonical sklearn Sweep Inputs

This notebook demonstrates the three operational survival modes used in practice:

1. Native survival data (`NATIVE`)
2. Auxiliary metric-derived survival framing (`AUXILIARY_METRIC`)
3. Auxiliary failure-derived survival framing (`AUXILIARY_FAILURE`)

For modes 2 and 3, we compose the canonical sklearn config from `examples/sklearn/config/default.yaml`, including the sweep defaults (`search/models` and `search/attacks`), and instantiate real sklearn/ART components.

### Scope and Guardrails

This notebook demonstrates survival-mode composition and data-shaping contracts.

- It is a mode-comparison notebook, not a benchmark for survival model quality.
- The goal is to clarify `NATIVE`, `AUXILIARY_METRIC`, and `AUXILIARY_FAILURE` wiring.
- Inputs are kept close to canonical sklearn configuration so downstream DVC and optimization flows remain reproducible.

For plugin API reference, see {doc}`/api/plugins/lifelines` and {doc}`/developers/experiment`.

In [1]:
from importlib import import_module
from pathlib import Path

import numpy as np
import pandas as pd
from hydra import compose, initialize_config_dir
from hydra.core.config_store import ConfigStore
from hydra.core.global_hydra import GlobalHydra
from hydra.utils import instantiate
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

from deckard.data import DataConfig
from deckard.plugins.lifelines.data import LifelinesDataConfig

PROJECT_ROOT = Path("../..").resolve()
CONFIG_DIR = PROJECT_ROOT / "examples" / "sklearn" / "config"


def reset_hydra_state() -> None:
    if GlobalHydra.instance().is_initialized():
        GlobalHydra.instance().clear()
    config_store = ConfigStore.instance()
    for key in list(config_store.repo.keys()):
        if key not in {"hydra", "_dummy_empty_config_.yaml"}:
            config_store.repo.pop(key, None)


reset_hydra_state()
with initialize_config_dir(version_base="1.3", config_dir=str(CONFIG_DIR)):
    cfg = compose(
        config_name="default",
        overrides=["score=classification", "model=rf", "attack=hsj"],
        return_hydra_config=True,
    )

print("Composed canonical sklearn config from:", CONFIG_DIR)
print("Model alias:", cfg.model.alias)
print("Attack alias:", cfg.attack.alias)
print("Search groups from defaults: search/models, search/attacks")
print(
    "Sweeper model params keys:",
    [k for k in cfg.hydra.sweeper.params.keys() if "model." in k][:3],
)
print(
    "Sweeper attack params keys:",
    [k for k in cfg.hydra.sweeper.params.keys() if "attack." in k][:3],
)

/Users/c.meyers/.pyenv/versions/3.10.20/envs/deckard/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Composed canonical sklearn config from: /Users/c.meyers/Documents/deckard/examples/sklearn/config
Model alias: rf
Attack alias: hsj
Search groups from defaults: search/models, search/attacks
Sweeper model params keys: ['++model.model_params.n_estimators', '++model.model_params.max_depth', '++model.model_params.max_leaf_nodes']
Sweeper attack params keys: ['++attack.attack_params.max_iter', '++attack.attack_params.max_eval', '++attack.attack_params.init_eval']


## Mode 1: NATIVE

`NATIVE` is used when the data is already in survival format with explicit duration and event columns.

Here we create a [LifelinesDataConfig](../api/modules) using the native constructor path ([from_data_and_model](../api/modules)).

In [2]:
base_native = DataConfig(
    name="lifelines_diabetes",
    target="E",
    classifier=False,
    data_params={},
)

native_cfg = LifelinesDataConfig.from_data_and_model(
    data_config=base_native,
    duration_col="T",
    event_col="E",
)

print("Mode:", native_cfg.mode)
print("is_native_survival_data:", native_cfg.is_native_survival_data())
print("duration/event:", native_cfg.duration_col, native_cfg.event_col)

Mode: LifelinesDataMode.NATIVE
is_native_survival_data: True
duration/event: T E


## Mode 2: AUXILIARY_METRIC (Real sklearn Model)

In `AUXILIARY_METRIC`, survival labels are derived from baseline model behavior on a standard dataset.

We instantiate the canonical model config (`model=rf`) from the composed sklearn defaults, train the real sklearn model, and convert prediction quality into surrogate survival columns.

In [3]:
# Instantiate the canonical sklearn model class directly from composed config.
model_module_name, model_class_name = cfg.model.name.rsplit(".", 1)
model_class = getattr(import_module(model_module_name), model_class_name)
sk_model = model_class(**dict(cfg.model.model_params))

X, y = make_classification(
    n_samples=160,
    n_features=8,
    n_informative=5,
    n_redundant=0,
    random_state=7,
    class_sep=1.2,
    flip_y=0.03,
    weights=[0.55, 0.45],
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=11, stratify=y
)
sk_model.fit(X_train, y_train)

y_pred = sk_model.predict(X_test)
benign_accuracy = float(accuracy_score(y_test, y_pred))

# Surrogate survival framing for mode 2:
# T_model is uncertainty-derived duration proxy, E_model marks benign failures.
if hasattr(sk_model, "predict_proba"):
    proba = sk_model.predict_proba(X_test)
    confidence = proba.max(axis=1)
else:
    confidence = np.where(y_pred == y_test, 0.9, 0.1)

T_model = (100.0 * (1.0 - confidence)).astype(float)
E_model = (y_pred != y_test).astype(int)

base_aux_metric = DataConfig(
    name="sklearn_make_classification",
    target="E_model",
    classifier=False,
    data_params={},
)
aux_metric_cfg = LifelinesDataConfig.from_auxiliary_metric(
    data_config=base_aux_metric,
    reference_metric="accuracy",
)

mode2_df = pd.DataFrame(
    {
        "T_model": T_model,
        "E_model": E_model,
    }
)

print("Mode:", aux_metric_cfg.mode)
print("has_auxiliary_metric:", aux_metric_cfg.has_auxiliary_metric())
print("Canonical model:", cfg.model.name)
print("Reference metric accuracy:", round(benign_accuracy, 4))
print("Mode 2 frame head:")
print(mode2_df.head(3))

Mode: LifelinesDataMode.AUXILIARY_METRIC
has_auxiliary_metric: True
Canonical model: sklearn.ensemble.RandomForestClassifier
Reference metric accuracy: 0.875
Mode 2 frame head:
   T_model  E_model
0     23.0        0
1     47.0        0
2     38.0        0


## Mode 3: AUXILIARY_FAILURE (Real ART Attack)

In `AUXILIARY_FAILURE`, survival labels are derived from failure outcomes.

Failure outcomes can come from attack traces or non-attack sources (for example, runtime incident rates). In this notebook we demonstrate the attack-backed path by instantiating `attack=hsj` and running a real ART HopSkipJump attack against the trained sklearn model.

In [4]:
import json

from art.estimators.classification.scikitlearn import ScikitlearnClassifier

attack_cfg = instantiate(cfg.attack)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

attack_model_module, attack_model_class = cfg.model.name.rsplit(".", 1)
attack_model_cls = getattr(import_module(attack_model_module), attack_model_class)
sk_model_for_attack = attack_model_cls(**dict(cfg.model.model_params))
sk_model_for_attack.fit(X_train_scaled, y_train)
art_classifier = ScikitlearnClassifier(
    model=sk_model_for_attack,
    clip_values=(0.0, 1.0),
)

attack_module_name, attack_class_name = cfg.attack.name.rsplit(".", 1)
attack_class = getattr(import_module(attack_module_name), attack_class_name)

demo_attack_params = dict(cfg.attack.attack_params)
demo_attack_params.update(
    {
        "max_iter": min(int(demo_attack_params.get("max_iter", 10)), 1),
        "max_eval": min(int(demo_attack_params.get("max_eval", 20)), 20),
        "init_eval": min(int(demo_attack_params.get("init_eval", 1)), 5),
        "init_size": min(int(demo_attack_params.get("init_size", 100)), 16),
        "verbose": False,
    }
)

try:
    attack_obj = attack_class(estimator=art_classifier, **demo_attack_params)
except TypeError:
    attack_obj = attack_class(classifier=art_classifier, **demo_attack_params)

x_eval = X_test_scaled[:12]
pred_clean = sk_model_for_attack.predict(x_eval)
x_adv = attack_obj.generate(x=x_eval)
pred_adv = sk_model_for_attack.predict(x_adv)

attack_success = (pred_adv != pred_clean).astype(int)
attack_success_rate = float(attack_success.mean())

T_attack = np.linalg.norm((x_adv - x_eval).reshape(len(x_eval), -1), axis=1).astype(
    float
)
E_attack = attack_success

base_aux_failure = DataConfig(
    name="sklearn_make_classification",
    target="E_attack",
    classifier=False,
    data_params={},
)
aux_failure_cfg = LifelinesDataConfig.from_auxiliary_failure(
    data_config=base_aux_failure,
    failure_profile={
        "alias": cfg.attack.alias,
        "name": cfg.attack.name,
        "attack_params": demo_attack_params,
    },
)

mode3_df = pd.DataFrame(
    {
        "T_attack": T_attack,
        "E_attack": E_attack,
    }
)

print("Mode:", aux_failure_cfg.mode)
print("has_auxiliary_failure:", aux_failure_cfg.has_auxiliary_failure())
print("Canonical attack:", cfg.attack.name)
print("Instantiated AttackConfig alias:", attack_cfg.alias)
print("Attack success rate:", round(attack_success_rate, 4))
print("Mode 3 frame head:")
print(mode3_df.head(3))

ARTIFACT_DIR = Path("build/notebook_artifacts/lifelines")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
mode2_df.to_csv(ARTIFACT_DIR / "mode2_aux_metric.csv", index=False)
mode3_df.to_csv(ARTIFACT_DIR / "mode3_aux_failure.csv", index=False)
summary = {
    "model_alias": cfg.model.alias,
    "attack_alias": cfg.attack.alias,
    "mode2_reference_accuracy": round(benign_accuracy, 4),
    "mode3_attack_success_rate": round(attack_success_rate, 4),
}
with open(ARTIFACT_DIR / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

Mode: LifelinesDataMode.AUXILIARY_FAILURE
has_auxiliary_failure: True
Canonical attack: art.attacks.evasion.HopSkipJump
Instantiated AttackConfig alias: hsj
Attack success rate: 1.0
Mode 3 frame head:
   T_attack  E_attack
0  0.436078         1
1  0.057730         1
2  0.390952         1
